# NB_patient_360 (Silver → Gold)

**Target table:** `adb_caresync.gold.patient_360`  
**Grain:** one row per patient  
**Load type:** full rebuild every run (`CREATE OR REPLACE TABLE ... AS SELECT`)

**How it works**
1. Start from `silver.patients` (the driving table — every patient is kept, no filters).
2. `LEFT JOIN` `silver.hospitals` to get the hospital name.
3. `LEFT JOIN` `silver.insurance_providers` to get the insurance provider name.
4. `LEFT JOIN` a per-patient summary of `silver.lab_results` (test counts and last test date).

LEFT JOINs are used so that a patient is never dropped just because a hospital, insurer or lab test is missing. Patients with no lab tests get `0` for the counts and `NULL` for `last_test_date`.

In [0]:
%sql
-- Step 1: summarise lab results per patient (one row per patient_id)
-- Step 2: join everything onto the patients table and write the gold table

CREATE OR REPLACE TABLE adb_caresync.gold.patient_360
COMMENT 'Gold: one row per patient with hospital, insurance and lab-test summary'
AS
WITH lab_summary AS (
  SELECT
    patient_id,
    COUNT(*)                              AS total_tests,
    COUNT_IF(result_status = 'Abnormal')  AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')  AS critical_test_count,
    MAX(test_date)                        AS last_test_date
  FROM adb_caresync.silver.lab_results
  GROUP BY patient_id
)
SELECT
  -- Patient details (pass-through)
  p.patient_id,
  p.first_name,
  p.last_name,
  CAST(p.date_of_birth AS DATE)                                              AS date_of_birth,  -- silver stores it as TIMESTAMP
  CAST(FLOOR(datediff(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,  -- age in years as of today
  p.gender,
  p.city,
  p.state,

  -- Hospital
  p.registered_hospital_id                     AS hospital_id,
  h.hospital_name,

  -- Insurance
  p.insurance_provider_id,
  ip.provider_name                             AS insurance_provider_name,

  -- Lab test summary (0 when the patient has no tests)
  CAST(COALESCE(l.total_tests, 0)         AS INT) AS total_tests,
  CAST(COALESCE(l.abnormal_test_count, 0) AS INT) AS abnormal_test_count,
  CAST(COALESCE(l.critical_test_count, 0) AS INT) AS critical_test_count,
  l.last_test_date                                -- NULL when the patient has no tests

FROM adb_caresync.silver.patients p
LEFT JOIN adb_caresync.silver.hospitals h
  ON p.registered_hospital_id = h.hospital_id
LEFT JOIN adb_caresync.silver.insurance_providers ip
  ON p.insurance_provider_id = ip.insurance_provider_id
LEFT JOIN lab_summary l
  ON p.patient_id = l.patient_id;

In [0]:
# %sql
# -- Row count should equal silver.patients, and patient_id should be unique
# SELECT
#   (SELECT COUNT(*) FROM adb_caresync.silver.patients) AS silver_patients,
#   COUNT(*)                                            AS gold_rows,
#   COUNT(DISTINCT patient_id)                          AS distinct_patients,
#   SUM(total_tests)                                    AS total_tests,
#   (SELECT COUNT(*) FROM adb_caresync.silver.lab_results) AS silver_lab_rows
# FROM adb_caresync.gold.patient_360;

In [0]:
# %sql
# SELECT * FROM adb_caresync.gold.patient_360 LIMIT 10;

In [0]:
# The table is built in a %sql cell, so there is no Python DataFrame.
# Read the gold table back into a DataFrame to get the row count.
updated_df = spark.table("adb_caresync.gold.patient_360")
record_count = updated_df.count()


dbutils.notebook.exit(str(record_count))